# Développez un assistant pour la recommandation d'évènements culturels

## Contexte

Vous êtes ingénieur data freelance spécialisé en intégration de modèles NLP et gestion de bases de données vectorielles. Vous travaillez pour l’entreprise Puls-Events, une société innovante dans le domaine de la gestion d’événements culturels.

 

L’entreprise Puls-Events propose une plateforme web permettant aux utilisateurs de découvrir et de suivre des événements culturels en temps réel. Elle collecte des informations à partir de diverses sources, comme Open Agenda, pour proposer des événements adaptés aux préférences de ses utilisateurs, filtrables par lieu et par période.

 

Elle souhaite tester l’intégration d’un chatbot intelligent, capable de fournir des recommandations personnalisées et augmentées par des réponses précises issues de données d’événements. 

 

Ce chatbot, soutenu par un système de génération de réponses augmentées (RAG), doit être capable d'extraire et d'analyser des informations à partir d'une base de données vectorielle.

 

Vous êtes chargé de réaliser un premier Proof of Concept (POC) pour démontrer la faisabilité de ce concept. 

 

Vous recevez un message de votre responsable technique : 

 

De  : Jérémy

A : moi

Objet :  version fonctionnelle du POC RAG

Salut,

 

Nous avons besoin que tu termines une version fonctionnelle du POC RAG. Il sera présenté en réunion avec les équipes produit et marketing pour vérifier l’intérêt du projet et envisager son déploiement à plus grande échelle.

 

Comme discuté ensemble, nous allons nous appuyer sur la source de données des événements publics Open Agenda. Tu pourras te limiter au périmètre géographique de ton choix. Veille cependant à ce qu’il y ait uniquement des événements récents de moins d’un an.

 

Ce qui est attendu :

* **Environnement virtuel** : Pour que le projet soit facilement reproductible, dépendances et documentation sous forme d’un readme.
* **Système RAG fonctionnel** : Code complet du système RAG versionné dans un repo git, avec intégration de LangChain, Mistral et Faiss. Le système doit être en mesure de reconstruire la base vectorielle sur demande (scripts de pré-processing et de vectorisation des données).
* **Rapport technique (PDF ou Word)** : Document entre 5 et 10 pages, expliquant l’architecture du système, tes choix techniques, les modèles sélectionnés, les résultats obtenus pour le POC et les recommandations pour la version finale.
* **Tests unitaires (Python)** : Prévois un fichier .py pour tester que les données intégrées dans la base vectorielle correspondent bien à des événements de moins d’un an dans la région géographique que tu auras sélectionnée.
* **Présentation (PowerPoint)** : Entre 10 et 15 slides pour exposer le concept du projet, les résultats du POC.
Démo live du système : Prépare une démo live pour illustrer les capacités du POC.
N’hésite pas si tu as besoin de précisions sur le périmètre ou les attentes. On compte sur toi pour que tout soit prêt dans les délais !

 

Ah oui, j’oubliais... Pour évaluer le système, nous aurons besoin de créer un jeu de données test annoté de questions/réponses. Ce jeu test sera utilisé pour mesurer la qualité des réponses par rapport aux réponses annotées.

 

Merci et bon courage !

 

Jérémy

## Étape 1 - Préparez l'environnement de travail



Création de l'environnement virtuel avec la commande suivante:

```
(base) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>python -m venv .venv_rag
```
 Désactivation de l'environnement de base puis activation de l'environnement virtuel `venv_rag`

 ```
(base) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>conda deactivate

C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>.venv_rag\Scripts\activate

(.venv_rag) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>

```
Installation des packages nécessaires:

```
(.venv_rag) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>pip install langchain

(.venv_rag) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>pip install faiss-cpu

(.venv_rag) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>pip install mistralai


```
https://github.com/mistralai/client-python



In [1]:
import sys
print(sys.executable)

C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11\.venv_rag\Scripts\python.exe


In [2]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import langchain
import faiss
import mistralai 
import numpy as np

In [4]:
# Vérification des versions
print(langchain.__version__)
print(faiss.__version__)
print(mistralai)

1.4.3
1.15.1
<module 'mistralai' (namespace) from ['C:\\Users\\sebas\\Documents\\cours_openclassrooms\\data_engineer\\projet_11\\.venv_rag\\Lib\\site-packages\\mistralai']>


Afin de vérifier la bonne installation de faiss, suivre la procédure présentée sous Github: https://github.com/facebookresearch/faiss/wiki/Getting-started

In [21]:
# Vérifier que FAISS est correctement installé et que la recherche CPU fonctionne
print("\n=== Test fonctionnel de FAISS (CPU) ===")

# Création de données fictives (100000 vecteurs de dimension 64, 10000 questions)
d = 64
nb = 100000
nq = 10000
np.random.seed(1234) # rendre reproductible
xb = np.random.random((nb, d)).astype("float32") 
xb[:, 0] += np.arange(nb) / 1000.
# xb représente la base de données contenant l’ensemble des vecteurs à indexer et dans laquelle nous effectuerons les recherches. Elle est de dimension nb × d.

xq = np.random.random((nq, d)).astype("float32")
xq[:, 0] += np.arange(nq) / 1000.
# xq désigne les vecteurs de requête pour lesquels nous devons trouver les plus proches voisins. Sa dimension est nq × d. 
# Si nous avons un seul vecteur de requête, alors nq = 1
    
# Initialiser un index CPU classique (IndexFlatL2)
index = faiss.IndexFlatL2(d) # IndexFlatL2 est un index FAISS qui s'exécute sur CPU par défaut.
print(f"Index est-il entraîné? {index.is_trained}")

# Ajouter les vecteurs à l'index
index.add(xb)
print(f"Nombre total de vecteurs indexés : {index.ntotal}")

# Effectuer une recherche de test
k=2 # voir les 2 voisins les plus proches
D, I = index.search(xb[:5], k) # Test de cohérence
print(I) # doit commencer par 0, correspond aux indices, classés du plus proche au plus éloigné
print(D) # contient les distances de chaque vecteur
D, I = index.search(xq, k) 
print("Voisins des 5 premiers vecteurs:\n", I[:5])   
print("Voisin des 5 derniers vecteurs:\n", I[-5:])
print("FAISS fonctionne correctement")


=== Test fonctionnel de FAISS (CPU) ===
Index est-il entraîné? True
Nombre total de vecteurs indexés : 100000
[[  0 393]
 [  1 555]
 [  2 304]
 [  3 173]
 [  4 288]]
[[1.1444092e-05 7.1751785e+00]
 [0.0000000e+00 6.3235626e+00]
 [7.6293945e-06 5.7964058e+00]
 [3.8146973e-06 7.2779045e+00]
 [0.0000000e+00 6.7638016e+00]]
Voisins des 5 premiers vecteurs:
 [[381 207]
 [526 911]
 [838 527]
 [196 184]
 [526 377]]
Voisin des 5 derniers vecteurs:
 [[ 9900 10500]
 [11055 10895]
 [11353 11103]
 [10571 10664]
 [ 9628  9554]]
FAISS fonctionne correctement


In [12]:
# source: https://github.com/facebookresearch/faiss/blob/main/tutorial/python/1-Flat.py
d = 64  # dimension
nb = 100000  # database size
nq = 10000  # nb of queries
np.random.seed(1234)  # make reproducible
xb = np.random.random((nb, d)).astype("float32")
xb[:, 0] += np.arange(nb) / 1000.0
xq = np.random.random((nq, d)).astype("float32")
xq[:, 0] += np.arange(nq) / 1000.0

index = faiss.IndexFlatL2(d)  # build the index
print(index.is_trained)
index.add(xb)  # add vectors to the index
print(index.ntotal)

k = 4  # we want to see 4 nearest neighbors
D, I = index.search(xb[:5], k)  # sanity check
print(I)
print(D)
D, I = index.search(xq, k)  # actual search
print(I[:5])  # neighbors of the 5 first queries
print(I[-5:])  # neighbors of the 5 last queries

True
100000
[[  0 393 363  78]
 [  1 555 277 364]
 [  2 304 101  13]
 [  3 173  18 182]
 [  4 288 370 531]]
[[1.1444092e-05 7.1751785e+00 7.2076340e+00 7.2511673e+00]
 [0.0000000e+00 6.3235626e+00 6.6845779e+00 6.7999382e+00]
 [7.6293945e-06 5.7964058e+00 6.3917274e+00 7.2815170e+00]
 [3.8146973e-06 7.2779045e+00 7.5279808e+00 7.6628418e+00]
 [0.0000000e+00 6.7638016e+00 7.2951088e+00 7.3688164e+00]]
[[ 381  207  210  477]
 [ 526  911  142   72]
 [ 838  527 1290  425]
 [ 196  184  164  359]
 [ 526  377  120  425]]
[[ 9900 10500  9309  9831]
 [11055 10895 10812 11321]
 [11353 11103 10164  9787]
 [10571 10664 10632  9638]
 [ 9628  9554 10036  9582]]


In [11]:
I[0]

array([381, 207, 210, 477])